# 03 – Premier modèle de prédiction : Elo

**Question :** avant un match, quelle est la probabilité que l'équipe à domicile gagne ?

**Modèles comparés**, du plus simple au plus élaboré :
1. *Pile ou face* : 50 % pour tout le monde.
2. *Fréquence historique* : part des victoires à domicile des 10 saisons précédentes.
3. *Elo* : une note par franchise, mise à jour après chaque match (`src/nba_analytics/models/elo.py`).
4. *Elo re-réglé* : mêmes calculs, mais l'avantage du terrain et la vitesse d'apprentissage sont re-choisis
   avant chaque saison, sur les 10 saisons précédentes uniquement.
5. *FiveThirtyEight* : la prévision publiée dans la source, comme point de comparaison.

**Règle d'or :** aucune prévision n'utilise d'information postérieure au match prédit. Le réglage se fait sur le passé,
la mesure sur des saisons jamais vues (2000 à 2015). Les tests `tests/test_models.py` vérifient cette règle.

⚠️ Une bonne prévision ne garantit aucun gain : voir la section 6.

## 0. Installation (Colab) ou détection du dépôt (local)

Dans Colab, la cellule clone le dépôt GitHub public, **sans jeton** : aucun secret n'est nécessaire pour lire.

In [ ]:
import os, sys, subprocess
from pathlib import Path

GITHUB_USER = "JohnGAB7"
REPO_NAME = "Sport-Analytics"
PROJECT_DIR = "nba"                  # le projet NBA est dans ce sous-dossier du dépôt
IN_COLAB = "google.colab" in sys.modules

if IN_COLAB:
    repo = Path("/content") / REPO_NAME
    if not repo.exists():
        # Dépôt public : clonage sans jeton (aucun secret dans le notebook ni dans .git/config).
        url = f"https://github.com/{GITHUB_USER}/{REPO_NAME}.git"
        subprocess.run(["git", "clone", "-q", url, str(repo)], check=True)
    repo = repo / PROJECT_DIR
    os.chdir(repo)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
else:
    # Exécution locale : on remonte jusqu'au dossier qui contient src/
    here = Path.cwd()
    repo = next(p for p in [here, *here.parents] if (p / "src" / "nba_analytics").exists())
    os.chdir(repo)

sys.path.insert(0, str(repo / "src"))
print("Dossier du dépôt :", Path.cwd().name)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from nba_analytics import quality
from nba_analytics.sources import fivethirtyeight as f38
from nba_analytics.models import backtest as bt, evaluation as ev
from nba_analytics.models.elo import EloParams, run_elo

raw = pd.read_csv(f38.download("data/raw/nbaallelo.csv"), dtype=str)
games, _ = quality.coerce_types(f38.to_games(raw))
games = bt.prepare(games)
print(len(games), "matchs |", f38.ATTRIBUTION)

## 1. Notre Elo est-il correctement codé ?

Avec les paramètres publiés par FiveThirtyEight (K = 20, avantage 100, retour à la moyenne 75 %), notre code doit
retrouver leurs probabilités. C'est un contrôle de l'implémentation, pas une mesure de qualité.

In [ ]:
check = games.assign(p_elo=run_elo(games, EloParams())["p_elo"]).merge(f38.forecasts(raw), on="game_id")
diff = (check["p_elo"] - check["p_538"]).abs()
print("Corrélation :", round(check[["p_elo", "p_538"]].corr().iloc[0, 1], 4))
print("Écart médian :", f"{diff.median():.2e}", "| matchs avec écart > 1 point de % :", int((diff > 0.01).sum()))

**Lecture :** les écarts de plus d'un point de pourcentage (environ 2 % des matchs) se situent tous entre 1948 et 1951
et en 1977-1981. Ce sont les années où des équipes d'autres ligues (NBL, ABA) ont rejoint la NBA : FiveThirtyEight leur
donne une note héritée de leur ancienne ligue, notre code leur donne une note de départ de 1300. Après 1981, aucun écart :
la période de test 2000-2015 n'est pas concernée.

## 2. Backtest 2000-2015

Le réglage « Elo fixe » est fait une fois sur 1977-1999. Le « Elo re-réglé » est refait avant chaque saison.
Cette cellule prend environ une minute.

In [ ]:
grid = bt.grid_search(games, range(1977, 2000))
best = grid.iloc[0]
fixed = EloParams(k=float(best.k), home_advantage=float(best.home_advantage),
                  carry_over=float(best.carry_over), use_margin=bool(best.use_margin))
print("Paramètres choisis sur 1977-1999 :", fixed)

wf, chosen = bt.walk_forward(games, range(2000, 2016))
pred = (bt.backtest(games, fixed, range(2000, 2016))
          .merge(wf[["game_id", "p_elo_wf"]], on="game_id")
          .merge(f38.forecasts(raw), on="game_id"))
chosen.T

## 3. Résultats

Plus le **Brier** et la **log-loss** sont bas, meilleur est le modèle. La **précision** (part des matchs où l'équipe
jugée favorite a gagné) est donnée pour information ; elle ne dit rien de la justesse des probabilités.
Pour « pile ou face », la précision n'a pas de sens (aucun favori).

In [ ]:
cols = {"p_coin": "Pile ou face", "p_home_rate": "Fréquence historique", "p_elo": "Elo fixe",
        "p_elo_wf": "Elo re-réglé", "p_538": "FiveThirtyEight (publié)"}
ev.summary(pred, list(cols)).rename(index=cols)

**Lecture :** Elo fait nettement mieux que les deux références simples. La version re-réglée corrige un biais
de la version fixe : l'avantage du terrain a diminué depuis les années 1980-90. Elle fait jeu égal avec la prévision
publiée par FiveThirtyEight. Prudence : les paramètres de FiveThirtyEight ont été choisis en connaissant toute
l'histoire, ce qui avantage leur colonne dans cette comparaison.

In [ ]:
seasons = bt.per_season(pred, ["p_home_rate", "p_elo", "p_elo_wf", "p_538"])
ax = seasons.rename(columns=cols).plot(figsize=(10, 3.8), marker="o", ms=3)
ax.set(title="Log-loss par saison (plus bas = mieux)", xlabel="saison", ylabel="log-loss")
plt.tight_layout(); plt.show()

## 4. Calibration

Si le modèle annonce 70 %, l'équipe doit gagner environ 70 % du temps. Points sur la diagonale = bien calibré.

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
for col, label in [("p_elo", "Elo fixe"), ("p_elo_wf", "Elo re-réglé")]:
    t = ev.calibration_table(pred[col], pred["home_win"])
    ax.plot(t["predicted"], t["observed"], marker="o", label=label)
ax.plot([0, 1], [0, 1], color="black", lw=0.8, ls="--")
ax.set(xlabel="probabilité annoncée", ylabel="fréquence observée", title="Calibration 2000-2015")
ax.legend(); plt.tight_layout(); plt.show()
ev.calibration_table(pred["p_elo_wf"], pred["home_win"])

## 5. Notes Elo à la fin des données (juin 2015)

In [ ]:
final = run_elo(games, EloParams(k=float(chosen.k.iloc[-1]), home_advantage=float(chosen.home_advantage.iloc[-1])))
last = games.assign(**final[["elo_home_pre", "elo_away_pre"]])
last = last[last.season == 2015]
notes = pd.concat([last[["game_date", "home_franchise", "elo_home_pre"]].set_axis(["date", "franchise", "elo"], axis=1),
                   last[["game_date", "away_franchise", "elo_away_pre"]].set_axis(["date", "franchise", "elo"], axis=1)])
notes.sort_values("date").groupby("franchise")["elo"].last().sort_values(ascending=False).round(0).head(10)

## 6. Ce que ces résultats permettent, et ne permettent pas

- **Ils permettent** de dire que le modèle donne des probabilités bien calibrées sur 2000-2015, saisons qu'il n'a pas vues.
- **Ils ne permettent pas** de dire qu'il gagnerait de l'argent : nous n'avons **aucune cote** dans les données.
  Un pari n'est rentable en moyenne que si `probabilité × cote > 1` (voir `ev.expected_value`), et les cotes
  intègrent la marge du bookmaker ainsi que des informations absentes d'Elo (blessures, repos, transferts).
  Elo est un modèle public et ancien : il est probable que les cotes en tiennent déjà compte (inférence, non mesurée).
- **Ils ne permettent pas** de prédire les matchs actuels : les données s'arrêtent en 2015.

Prochaines étapes nécessaires avant tout pari réel :
1. une source licite de résultats récents ; 2. une source licite de cotes historiques ;
3. un backtest contre ces cotes, marge incluse ; 4. une saison de paris fictifs enregistrés **avant** les matchs.